# 08 - Held-out mechanism evidence packs

v0.6 asks a harder question than whether an intervention works on the examples used to discover it: **does a frozen candidate survive examples it was not selected on?**

This notebook uses a deliberately adversarial synthetic model. Discovery examples truly use `route_a`; validation examples truly use `route_b`. A correct evidence framework should discover `route_a`, look excellent in-sample, and then reject the candidate on held-out data.

## Experiment contract

The protocol is:

```text
discovery examples -> candidate + donor fitting -> freeze
                                              |
validation examples --------------------------+-> causal test
```

Validation examples never enter candidate selection or mean-donor estimation.

In [ ]:
import tempfile
from pathlib import Path

import torch

from neuros_mechint.adapters import PyTorchAdapter
from neuros_mechint.benchmarks import (
    DiscoveryShiftMLP,
    EvidenceExample,
    EvidencePackPolicy,
    EvidencePackSpec,
    EvidenceSplit,
    discover_ablation_effect_candidate,
    read_evidence_pack_artifact,
    run_adapter_evidence_pack,
    write_evidence_pack_artifact,
)
from neuros_mechint.core import EvidenceTier, OutputMetric

In [ ]:
model = DiscoveryShiftMLP()
adapter = PyTorchAdapter(model)
metric = OutputMetric(lambda output: output.mean(), name="mean_output")

examples = (
    EvidenceExample("discover-1", torch.tensor([[1.0, 0.0]]), EvidenceSplit.DISCOVERY),
    EvidenceExample("discover-2", torch.tensor([[2.0, 0.0]]), EvidenceSplit.DISCOVERY),
    EvidenceExample("validate-1", torch.tensor([[0.0, 1.0]]), EvidenceSplit.VALIDATION),
    EvidenceExample("validate-2", torch.tensor([[0.0, 2.0]]), EvidenceSplit.VALIDATION),
)
targets = ("route_a", "route_b", "nuisance")

## Candidate discovery sees only the discovery split

We use the Research-maturity single-target ablation ranking. It is a candidate generator, not a claim that the highest-ranked component is universally causal.

In [ ]:
seen_by_discovery = []

def discover(adapter_, discovery_examples, all_targets):
    seen_by_discovery.extend(item.example_id for item in discovery_examples)
    return discover_ablation_effect_candidate(
        adapter_,
        discovery_examples,
        all_targets,
        metric=metric,
        k=1,
        name="discovery-top-route",
    )

In [ ]:
result = run_adapter_evidence_pack(
    spec=EvidencePackSpec(
        pack_id="tutorial-held-out-shift",
        model_id="DiscoveryShiftMLP",
        model_revision="synthetic-v1",
        dataset_id="synthetic:discovery-validation-shift",
        dataset_revision="synthetic-v1",
        metric_name=metric.name,
        target_universe=targets,
        discovery_method="single-target-zero-ablation",
        intervention_baselines=("zero", "mean"),
        random_trials=100,
        evidence_tier=EvidenceTier.SCIENTIFIC_SYNTHETIC,
    ),
    adapter=adapter,
    metric=metric,
    examples=examples,
    discover_candidate=discover,
    pack_policy=EvidencePackPolicy(bootstrap_samples=200),
)

In [ ]:
print("discovery saw:", seen_by_discovery)
print("candidate:", result.candidate.targets)
print("discovery pass rate:", result.discovery_aggregate.pass_rate)
print("validation pass rate:", result.validation_aggregate.pass_rate)
print("promotion:", result.promotion.passed)
print("reasons:", result.promotion.reasons)
print("mean donors:", dict(result.mean_ablation_references))

Expected interpretation:

- discovery sees only `discover-1` and `discover-2`;
- `route_a` is the correct discovery candidate;
- discovery faithfulness is strong;
- held-out faithfulness fails because the validation mechanism moved to `route_b`;
- the candidate is rejected.

That rejection is the success condition for this benchmark.

## Negative results are still reproducible artifacts

Promotion and publication readiness are separate. This synthetic study pins its model/data revisions, so the rejected mechanism can still be serialized and integrity-checked.

In [ ]:
assert result.publication_ready

with tempfile.TemporaryDirectory() as directory:
    artifact = write_evidence_pack_artifact(result, Path(directory) / "evidence.json")
    loaded = read_evidence_pack_artifact(artifact)
    print(loaded["study_fingerprint"] == result.study_fingerprint)
    print(loaded["promotion"]["passed"])

## What to carry into a real model

For a TransformerLens, NNsight, SAELens, circuit-tracer, NeuroFM, or ordinary PyTorch study:

1. pin model/data/tokenizer/SAE/transcoder revisions;
2. define the metric and target universe before discovery;
3. content-hash and reserve validation examples;
4. discover the candidate on discovery only;
5. freeze the candidate and perturbation donors;
6. run held-out interventions and matched controls;
7. keep failures and invalid cases;
8. publish the evidence artifact, not only the headline number.

The next scientific layer is replication across model seeds, sessions, subjects, and datasets.